In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import sklearn
sklearn.set_config(display='text')

베르누이 나이브 베이즈(Bernoulli Naive Bayes)

분류 데이터의 특징이 0 또는 1로 표현되었을 때 데이터의 출현 여부에 따라서 0 또는 1로 구분되는 데이터에 사용한다.

이메일 제목과 레이블(스팸 메일 여부)를 활용해 베르누이 나이브 베이즈 분류로 스팸 메일을 분류한다.

데이터 획득  
간단한 스팸 메일 분류를 위해 다음과 같이 이메일 제목과 레이블이 붙어있는 데이터를 사용한다.  
email title: 이메일 제목, spam: 스팸 메일 여부(True => 스팸 메일, False => 일반 메일)

In [2]:
# 학습 데이터
email_list = [
    {'email title' : 'free game only today', 'spam' : True},
    {'email title' : 'cheapest flight deal', 'spam' : True},
    {'email title' : 'limited time offer only today only today', 'spam' : True},
    {'email title' : 'today meeting schedule', 'spam' : False},
    {'email title' : 'your flight schedule attached', 'spam' : False},
    {'email title' : 'your credit card statement', 'spam' : False}
]

# 테스트 데이터
test_email_list = [
    {'email title' : 'free flight offer', 'spam' : True},
    {'email title' : 'hey traveler free flight deal', 'spam' : True},
    {'email title' : 'limited free game offer', 'spam' : True},
    {'email title' : 'today flight schedule', 'spam' : False},
    {'email title' : 'your credit card attached', 'spam' : False},
    {'email title' : 'free credit card offer only today', 'spam' : False}
]

학습 데이터 준비

In [3]:
df = pd.DataFrame(email_list)
df

,email title,spam
0,free game only today,True
1,cheapest flight deal,True
2,limited time offer only today only today,True
3,today meeting schedule,False
4,your flight schedule attached,False
5,your credit card statement,False


학습 데이터 다듬기  
사이킷런의 베르누이 나이브 베이즈 분류기는 숫자 데이터(0 또는 1)만 다룰수있기 때문에 True를 1로 False를 0으로 치환한 'label'이라는 파생 변수를 추가한다.

In [4]:
df['label'] = df.spam.map({True: 1, False: 0})
df

,email title,spam,label
0,free game only today,True,1
1,cheapest flight deal,True,1
2,limited time offer only today only today,True,1
3,today meeting schedule,False,0
4,your flight schedule attached,False,0
5,your credit card statement,False,0


학습 데이터를 피쳐와 레이블로 분리한다.

In [5]:
x = df['email title'] # 피쳐
x

0                        free game only today
1                        cheapest flight deal
2    limited time offer only today only today
3                      today meeting schedule
4               your flight schedule attached
5                  your credit card statement
Name: email title, dtype: str

In [22]:
y_train = df.label # 레이블
y_train

0    1
1    1
2    1
3    0
4    0
5    0
Name: label, dtype: int64

모든 이메일 제목을 공백을 경계로 연결해서 1개의 문자열로 만들고 오름차준 정렬된 리스트로 변환한다.

In [7]:
# string = 'free game only today cheapest flight deal limited time offer only today only today today meeting schedule your flight schedule attached ' \
#     'your credit card statement'
string = ' '.join(x)
print(string)
string = set(string.split())
print(string)
print(len(string))
string = list(string)
print(string)
string.sort()
print(string)

free game only today cheapest flight deal limited time offer only today only today today meeting schedule your flight schedule attached your credit card statement
{'cheapest', 'attached', 'credit', 'meeting', 'schedule', 'offer', 'card', 'only', 'today', 'time', 'limited', 'your', 'deal', 'statement', 'free', 'game', 'flight'}
17
['cheapest', 'attached', 'credit', 'meeting', 'schedule', 'offer', 'card', 'only', 'today', 'time', 'limited', 'your', 'deal', 'statement', 'free', 'game', 'flight']
['attached', 'card', 'cheapest', 'credit', 'deal', 'flight', 'free', 'game', 'limited', 'meeting', 'offer', 'only', 'schedule', 'statement', 'time', 'today', 'your']


이메일 제목으로 학습을 진행하고 레이블을 이용해서 스팸 메일 여부를 판단한다.

베르누이 나이브 베이즈의 입력 데이터는 0 또는 1로만 구성된 고정된(동일한) 크기의 벡터이어야 한다.

In [8]:
# 모든 데이터에 출현한 단어의 개수 만큼의 크기를 가지는 벡터를 만들고 고정된 벡터로 표현하기 위해 import 한다.
from sklearn.feature_extraction.text import CountVectorizer

CountVectorizer 클래스 객체는 인수로 넘긴 문자열에 출현한 모든 단어를 오름차순으로 정렬해서 단어의 위치로 행렬을 만든다.  
특정 단어가 출현할 경우 출현한 단어의 개수를 리턴하고 출현하지 않으면 0을 리턴한다.  
binary 속성의 기본값은 None이고 출현한 단어의 개수를 리턴하고 True로 변경하면 같은 단어가 여러번 출현하더라도 무조건 1로 리턴한다.

In [17]:
cv = CountVectorizer(binary=True) # CountVectorizer 클래스 객체를 만든다.
# x_train = cv.fit(x) # CountVectorizer 클래스 객체를 학습한다.
# x_train = cv.transform(x) # CountVectorizer 클래스 객체의 학습 결과를 적용한다.
x_train = cv.fit_transform(x) # CountVectorizer 클래스 객체를 학습하고 학습 결과를 적용한다.
print(x_train)
encoding = x_train.toarray() # CountVectorizer 클래스 객체의 학습 결과를 numpy 배열로 만든다.
# print(type(encoding))
print(encoding)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 23 stored elements and shape (6, 17)>
  Coords	Values
  (0, 6)	1
  (0, 7)	1
  (0, 11)	1
  (0, 15)	1
  (1, 2)	1
  (1, 5)	1
  (1, 4)	1
  (2, 11)	1
  (2, 15)	1
  (2, 8)	1
  (2, 14)	1
  (2, 10)	1
  (3, 15)	1
  (3, 9)	1
  (3, 12)	1
  (4, 5)	1
  (4, 12)	1
  (4, 16)	1
  (4, 0)	1
  (5, 16)	1
  (5, 3)	1
  (5, 1)	1
  (5, 13)	1
[[0 0 0 0 0 0 1 1 0 0 0 1 0 0 0 1 0]
 [0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 1 0 1 1 0 0 1 1 0]
 [0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 1 0]
 [1 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 1]
 [0 1 0 1 0 0 0 0 0 0 0 0 0 1 0 0 1]]


위의 넘파이 배열에서 볼 수 있듯이 이메일 제목에서 총 17개의 단어가 발결되어 각 이메일 제목이 17개 크기의 벡터로 인코딩(포현)된 것을 확인할 수 있다.  
binary=True 속성을 지정해서 베르누이 나이브 베이즈의 입력으로 사용하기 위해 이메일에 중복되는 단어가 있더라도 출현한 횟수로 표현하는 것이 아니고 1로 표현된 것을 알 수 있다.

In [18]:
# inverse_transform() 메소드로 고정된 크기의 벡터에 포함된 단어를 확인할 수 있다.
for s in cv.inverse_transform(x_train):
    print(s)

['free' 'game' 'only' 'today']
['cheapest' 'flight' 'deal']
['only' 'today' 'limited' 'time' 'offer']
['today' 'meeting' 'schedule']
['flight' 'schedule' 'your' 'attached']
['your' 'credit' 'card' 'statement']


In [20]:
# get_feature_names_out() 메소드로 고정된 벡터의 각 열이 어떤 단어를 의미하는지 확인할 수 있다.
print(cv.get_feature_names_out())

['attached' 'card' 'cheapest' 'credit' 'deal' 'flight' 'free' 'game'
 'limited' 'meeting' 'offer' 'only' 'schedule' 'statement' 'time' 'today'
 'your']


베르누이 나이브 베이즈 모델 학습하기

사이킷런의 베르누이 나이브 베이즈 분류기는 기본적으로 라플라스 스무딩 기능을 지원하므로 학습 데이터에 없는 단어가 테스트 데이터에 있어도 분류를 진행한다.  

라플라스 스무딩(Laplace Smoothing)  
0이라는 수는 곱셈과 나눗셈을 무력화시키는 값이므로 그 전에 아무리 의미있는 값이 도출된다 하더라도 마지막에 0을 곱해버리면 갑은 0이 나오게 된다. 이런 경우가 상당히 빈번하기 대문에 0이 아닌 최소값으로 보정을 하는데 이를 라플라스 스무딩이라 한다.

In [21]:
# 베르누이 나이브 베이즈 모델을 사용하기 위해 import 한다.
from sklearn.naive_bayes import BernoulliNB

In [23]:
# 베르누이 나이브 베이즈 모델을 만들고 학습시킨다.
model = BernoulliNB().fit(x_train, y_train)

학습된 모델을 테스트하기 위해서 테스트 데이터를 준비하고 다듬기

In [24]:
# 테스트 데이터를 데이터프레임으로 만든다.
df = pd.DataFrame(test_email_list)
# 사이킷런의 베르누이 나이브 베이즈 분류기는 숫자 데이터(0 또는 1)만 다룰수있기 때문에 True를 1로 False를 0으로 치환한 'label'이라는 파생 변수를 추가한다.
df['label'] = df.spam.map({True: 1, False: 0})
df

,email title,spam,label
0,free flight offer,True,1
1,hey traveler free flight deal,True,1
2,limited free game offer,True,1
3,today flight schedule,False,0
4,your credit card attached,False,0
5,free credit card offer only today,False,0


In [25]:
# 테스트 데이터를 피쳐와 레이블로 분리한다.
x = df['email title'] # 피쳐
x

0                    free flight offer
1        hey traveler free flight deal
2              limited free game offer
3                today flight schedule
4            your credit card attached
5    free credit card offer only today
Name: email title, dtype: str

In [26]:
y_test = df.label # 레이블
y_test

0    1
1    1
2    1
3    0
4    0
5    0
Name: label, dtype: int64

모델 테스트

In [28]:
# CountVectorizer 클래스 객체는 학습 데이터를 전처리 할 때 이미 학습이 진행된 상태이므로 모델 테스트시에는 적용만 시키면 된다.
x_test = cv.transform(x)
print(x_test)
print(x_test.toarray())

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 23 stored elements and shape (6, 17)>
  Coords	Values
  (0, 5)	1
  (0, 6)	1
  (0, 10)	1
  (1, 4)	1
  (1, 5)	1
  (1, 6)	1
  (2, 6)	1
  (2, 7)	1
  (2, 8)	1
  (2, 10)	1
  (3, 5)	1
  (3, 12)	1
  (3, 15)	1
  (4, 0)	1
  (4, 1)	1
  (4, 3)	1
  (4, 16)	1
  (5, 1)	1
  (5, 3)	1
  (5, 6)	1
  (5, 10)	1
  (5, 11)	1
  (5, 15)	1
[[0 0 0 0 0 1 1 0 0 0 1 0 0 0 0 0 0]
 [0 0 0 0 1 1 1 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 1 1 1 0 1 0 0 0 0 0 0]
 [0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 1 0]
 [1 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1]
 [0 1 0 1 0 0 1 0 0 0 1 1 0 0 0 1 0]]


In [29]:
# predict() 메소드의 인수로 테스트 데이터의 피쳐를 넘겨서 예측값을 계산한다.
predict = model.predict(x_test)
print(predict)
# accuracy_score() 함수의 인수로 테스트 데이터의 레이블(실제값)과 예측값을 넘겨서 정확도를 계산한다.
accuracy = accuracy_score(y_test, predict)
print('정확도: {:6.2%}'.format(accuracy))

[1 1 1 0 0 1]
정확도: 83.33%


In [30]:
# confusion_matrix() 함수의 인수로 테스트 데이터의 레이블(실제값)과 예측값을 넘겨서 혼동 행렬을 출력한다.
print(confusion_matrix(y_test, predict))

[[2 1]
 [0 3]]


In [31]:
# classification_report() 함수의 인수로 테스트 데이터의 레이블(실제값)과 예측값을 넘겨서 분류 리포트를 출력한다.
print(classification_report(y_test, predict))

              precision    recall  f1-score   support

           0       1.00      0.67      0.80         3
           1       0.75      1.00      0.86         3

    accuracy                           0.83         6
   macro avg       0.88      0.83      0.83         6
weighted avg       0.88      0.83      0.83         6

